# Overall TF expression versus TF activity

This standalone figure notebook reads the saved summary from `original_10_OVERALL_all_context_features_pretune_LOOCV.R`; it does not refit models or perform inference. It displays expression and activity AUPR/AUC with the activity-minus-expression difference. The two metric panels are arranged side by side.

Edit the input/output paths and `panel_width`/`panel_height` in the settings cell as needed.

In [ ]:
suppressPackageStartupMessages({
  library(ggplot2)
  library(dplyr)
  library(tidyr)
  library(egg)
  #library(showtext)
  #library(sysfonts)
})

# Input generated by original_10_OVERALL_all_context_features_pretune_LOOCV.R
analysis_dir <- "/project/zduren/durenlab/naqing_palmetto/REGA_separate/07_tf/results/classification_overall_alpha050"
summary_file <- file.path(analysis_dir, "OVERALL_classification_summary.tsv")

# Figure output settings
figure_dir <- file.path(analysis_dir, "figures_only_notebook")
dir.create(figure_dir, recursive = TRUE, showWarnings = FALSE)
save_figures <- TRUE
png_dpi <- 300
panel_width <- 4.3  # inches per panel
panel_height <- 4.3 # inches
font_family <- "sans"
#showtext_auto(enable = TRUE)

expr_color <- "#E76F51"
act_color  <- "#159D9C"

cat("Reading summary:\n", summary_file, "\n")
cat("Saving figures to:\n", figure_dir, "\n")

In [ ]:
summary_df <- read.delim(
  summary_file,
  check.names = FALSE,
  stringsAsFactors = FALSE
)

required_columns <- c(
  "Expression_AUPR", "Activity_AUPR", "Activity_minus_Expression_AUPR",
  "Expression_AUC", "Activity_AUC", "Activity_minus_Expression_AUC"
)
missing_columns <- setdiff(required_columns, names(summary_df))
if (length(missing_columns) > 0) {
  stop("Missing required summary columns: ", paste(missing_columns, collapse = ", "))
}
if (nrow(summary_df) != 1) {
  stop("Expected one overall classifier summary row; found ", nrow(summary_df))
}

plot_df <- bind_rows(
  data.frame(
    Metric = "AUPR",
    Data = c("Expression", "Activity"),
    Value = c(summary_df$Expression_AUPR[1], summary_df$Activity_AUPR[1]),
    Delta = summary_df$Activity_minus_Expression_AUPR[1]
  ),
  data.frame(
    Metric = "AUC",
    Data = c("Expression", "Activity"),
    Value = c(summary_df$Expression_AUC[1], summary_df$Activity_AUC[1]),
    Delta = summary_df$Activity_minus_Expression_AUC[1]
  )
) %>%
  mutate(
    Metric = factor(Metric, levels = c("AUPR", "AUC")),
    Data = factor(Data, levels = c("Expression", "Activity")),
    value_label = sprintf("%.3f", Value)
  )

plot_df

In [ ]:
make_metric_plot <- function(metric_name, y_label) {
  d <- filter(plot_df, Metric == metric_name)
  delta_value <- d$Delta[1]

  ggplot(d, aes(x = Data, y = Value, group = 1)) +
    geom_line(color = "grey55", linewidth = 0.7) +
    geom_point(aes(color = Data), size = 3.5) +
    geom_text(aes(label = value_label), vjust = -1.0, size = 4.2, family = font_family) +
    annotate(
      "text", x = 1.5, y = 0.08,
      label = sprintf("Delta (Activity - Expression) = %+.3f", delta_value),
      size = 3.8, family = font_family
    ) +
    scale_color_manual(values = c("Expression" = expr_color, "Activity" = act_color)) +
    scale_y_continuous(limits = c(0, 1.08), breaks = seq(0, 1, by = 0.2), expand = expansion(mult = c(0, 0.02))) +
    labs(x = NULL, y = y_label) +
    theme_article(base_size = 15) +
    theme(
      text = element_text(family = font_family),
      legend.position = "none",
      plot.margin = margin(10, 10, 10, 10),
      axis.text.x = element_text(size = 12)
    )
}

p_aupr <- make_metric_plot("AUPR", "AUPR")
p_auc  <- make_metric_plot("AUC", "ROC AUC")

# Each panel uses panel_width x panel_height; the combined output is two panels wide.
p_side_by_side <- egg::ggarrange(p_aupr, p_auc, ncol = 2)
options(repr.plot.width = 2 * panel_width, repr.plot.height = panel_height, repr.plot.res = 120)
print(p_side_by_side)

if (save_figures) {
  ggsave(file.path(figure_dir, "overall_AUPR_AUC_expression_activity.pdf"), p_side_by_side, width = 2 * panel_width, height = panel_height, units = "in", limitsize = FALSE)
  ggsave(file.path(figure_dir, "overall_AUPR_AUC_expression_activity.png"), p_side_by_side, width = 2 * panel_width, height = panel_height, units = "in", dpi = png_dpi, limitsize = FALSE)
}